In [13]:
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)
# 2. LOAD DATASET
df = pd.read_csv("historical_data.csv")
print("========== INITIAL DATA OVERVIEW ==========")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
# 3. DATA PREPROCESSING
for col in df.columns:
    if 'date' in col.lower():
        df[col] = pd.to_datetime(df[col], errors='coerce')
df_numeric = df.select_dtypes(include=[np.number])
df[df_numeric.columns] = df_numeric.fillna(df_numeric.mean())
df.drop_duplicates(inplace=True)
# OUTLIER HANDLING (IQR METHOD)
Q1 = df_numeric.quantile(0.25)
Q3 = df_numeric.quantile(0.75)
IQR = Q3 - Q1
df = df[~((df_numeric < (Q1 - 1.5 * IQR)) |
          (df_numeric > (Q3 + 1.5 * IQR))).any(axis=1)]
print("\nCLEANED DATA:")
print("New Shape:", df.shape)
# 4. EXPLORATORY DATA ANALYSIS
import os
os.makedirs("outputs", exist_ok=True)
#1. Missing Values Heatmap
plt.figure()
sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.savefig("outputs/missing_values.png")
plt.close()

#2. Distribution Plots
df_numeric = df.select_dtypes(include=[np.number])
df_numeric.hist(bins=30)
plt.suptitle("Feature Distributions")
plt.savefig("outputs/distributions.png")
plt.close()
# 3. Correlation Heatmap
plt.figure(figsize=(10,8))
sns.heatmap(df_numeric.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Matrix")
plt.savefig("outputs/correlation.png")
plt.close()
# 4. Boxplot (Outliers)
plt.figure()
df_numeric.boxplot()
plt.xticks(rotation=45)
plt.title("Boxplot for Outlier Detection")
plt.savefig("outputs/boxplot.png")
plt.close()
# 5. Skewness
plt.figure()
skewness = df_numeric.skew()
sns.barplot(x=skewness.index, y=skewness.values)
plt.xticks(rotation=45)
plt.title("Skewness of Features")
plt.savefig("outputs/skewness.png")
plt.close()
# 6. Statistical Summary
summary = df_numeric.describe()
summary.to_csv("outputs/statistical_summary.csv")

# 7. Time Series Plot
date_col = None
for col in df.columns:
    if 'date' in col.lower():
        date_col = col
        break
if date_col:
    df = df.sort_values(by=date_col)
    col = df_numeric.columns[0]
    plt.figure()
    plt.plot(df[date_col], df[col])
    plt.title(f"{col} Over Time")
    plt.xlabel("Date")
    plt.ylabel(col)
    plt.savefig("outputs/time_series.png")
    plt.close()
# 8. Pairplot
sns.pairplot(df_numeric.sample(min(500, len(df_numeric))))
plt.savefig("outputs/pairplot.png")
plt.close()
# 5. SAVE CLEANED DATA
df.to_csv("outputs/processed_data.csv", index=False)

========== INITIAL DATA OVERVIEW ==========
Shape: (211224, 16)

Columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']

Missing Values:
 Account             0
Coin                0
Execution Price     0
Size Tokens         0
Size USD            0
Side                0
Timestamp IST       0
Start Position      0
Direction           0
Closed PnL          0
Transaction Hash    0
Order ID            0
Crossed             0
Fee                 0
Trade ID            0
Timestamp           0
dtype: int64

CLEANED DATA:
New Shape: (49959, 16)
